In [ ]:
# https://chatgpt.com/c/68bc2472-baec-8331-88cd-1410188ed832

# Conda activate ITP259, zive_dask.

# Patobulintas variantas su išsamesne dokumentacija ir klaidų tvarkymu.
# Grafikos skriptas nepateikia.
# Pritaikytas naujam Excel failui, kuris turi šiek tiek kitokias meta duomenų struktūras (quality, tag, noni, mark, N, S, V, comment).
# pritaikytas kitokiam įrašo failo formatui: basename

"""
EKG triukšmų analizė su U-Net (Keras) modeliu.
- Patikrina modelio/žurnalų suderinamumą.
- Užkrauna modelį ir įrašų sąrašą iš Excel (ignoruoja tag == 9999).
- Filtruoja signalą, randa outliers/rdropouts, U-Net triukšmus.
- Skaičiuoja metrikas ir išsaugo .txt + .xlsx + summary.txt.

Reikalingi moduliai:
- zive_util_ml.get_ecg_signal
- use_ecg_denoising_util.{bandpass_filter, find_outliers_rdropouts, merge_lists_of_tuples, run_ecg_denoising_pipeline}
"""

from __future__ import annotations

import json
import logging
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import keras  
import numpy as np
import pandas as pd
import neurokit2 as nk  

import sys

# === Išoriniai moduliai (lygiagretus aplankas) =================================
PARALLEL_PATH = Path().resolve().parent / "SUPL_FUNCTIONS"
sys.path.append(str(PARALLEL_PATH))

from project_util import find_project_root_by_name
from zive_util_ml import get_ecg_signal  # noqa: E402
from use_ecg_denoising_util import (  # noqa: E402
    find_outliers_rdropouts,
    merge_lists_of_tuples,
    run_ecg_denoising_pipeline,
    ecg_filter
)


# === Konfigūracija ==============================================================

PROJECT_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PROJECT_ROOT = find_project_root_by_name(target="PROJECT_TRAIN_UNET", start=PROJECT_DIR)
print("\nPROJECT ROOT DIR:", PROJECT_ROOT)
print("PROJECT DIR:", PROJECT_DIR)

MODEL_DIR = PROJECT_ROOT / "MODEL_UNET"
# LIST_DIR = PROJECT_DIR / "AtsisiuntimasZiveDuomenu"/ "PseudoAnnotated"
# REC_DIR = PROJECT_DIR / "AtsisiuntimasZiveDuomenu"/ "PseudoAnnotated"

LIST_DIR = PROJECT_DIR / "AtsisiuntimasZiveDuomenu"/ "DuomenysTestui"
REC_DIR = PROJECT_DIR / "AtsisiuntimasZiveDuomenu"/ "DuomenysTestui"

MODEL_FILE_NAME = "resunet_ecg_1024_0_5_3_7.keras"
MARKER = MODEL_FILE_NAME.removeprefix("resunet_ecg").removesuffix(".keras")  # -> "_1024_0_5_3_7"
NOISE_LOG_BASENAME = f"noise_log{MARKER}"

EXCEL_NAME = "visi_zive_irasai_atrankai._modif_v1 - Darb - testui.xlsx"
# EXCEL_NAME = "Pseudo_annotated.xlsx"

FS = 200
SEGMENT_LENGTH = 1024
OVERLAP = 0.5
CONFIG = {"FS": FS, "SEGMENT_LENGTH": SEGMENT_LENGTH, "OVERLAP": OVERLAP}

THRESHOLD = 0.08  # U-Net residual noise threshold

# ECG įrašo filtravimui
fp = {  'type': 'lowpass',
            'method':'butterworth',
            'order':5,
            'sampling_rate':FS,
            'lowcut':0.5,
            'highcut':90 }



# === Pagalbinės funkcijos =======================================================


def setup_logging() -> None:
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S",
        stream=sys.stdout,   # <- key change, messages redirected not to stderr, but to stdout
        force=True,          # <- helps if logging already configured elsewhere
    )

def check_consistency(model_file_name: str, noise_log_basename: str, config: dict) -> None:
    """Patikrina failų sufiksų ir parametrų suderinamumą."""
    model_suffix = model_file_name.split("resunet_ecg")[-1].replace(".keras", "")
    noise_suffix = noise_log_basename.split("noise_log")[-1]

    if model_suffix != noise_suffix:
        raise ValueError(f"Suffix mismatch:\n  model: {model_suffix}\n  noise: {noise_suffix}")

    seg_str = str(config["SEGMENT_LENGTH"])
    if seg_str not in model_suffix:
        raise ValueError(f"SEGMENT_LENGTH mismatch: expected '{seg_str}' in '{model_suffix}'")

    overlap_str = str(config["OVERLAP"]).replace(".", "_")
    if overlap_str not in model_suffix:
        raise ValueError(f"OVERLAP mismatch: expected '{overlap_str}' in '{model_suffix}'")

    logging.info("Suderinamumas OK | sufiksas:%s | SEGMENT_LENGTH:%s | OVERLAP:%s",
                 model_suffix, seg_str, overlap_str)


def load_model_checked(model_path: Path, segment_len: int) -> keras.Model:
    """Užkrauna Keras modelį ir patikrina input shape == (segment_len, 1)."""
    logging.info("Kraunamas modelis: %s", model_path)
    try:
        model = keras.models.load_model(str(model_path))
    except Exception as exc:  # noqa: BLE001
        raise ValueError(f"Failed to load model from {model_path}: {exc}") from exc

    if not hasattr(model, "input_shape") or model.input_shape is None:
        raise ValueError("Loaded model has no valid input_shape")

    expected = (segment_len, 1)
    if model.input_shape[1:] != expected:
        raise ValueError(f"Model expects {model.input_shape[1:]}, script uses {expected}")

    logging.info("Modelis OK, input_shape: %s", model.input_shape)
    return model


def get_ecg_noise_indices_annotated_ext(json_path: Path) -> Optional[List[Tuple[int, int]]]:
    """
    Grąžina [(startIndex, endIndex), ...] iš JSON 'noises_annotated'.
    Jei failo nėra ar raktas neegzistuoja – grąžina None (aiškus signalas).
    """
    if not json_path.exists():
        return None

    with open(json_path, "r", encoding="utf-8", errors="ignore") as f:
        data = json.load(f)

    items = data.get("noises_annotated")
    if not isinstance(items, list):
        return None

    out: List[Tuple[int, int]] = []
    for it in items:
        try:
            out.append((int(it["startIndex"]), int(it["endIndex"])))
        except (KeyError, TypeError, ValueError):
            # praleidžiam brokuotą įrašą, bet netrukdom kitų
            continue
    return out if out else None


def safe_int(value: object) -> Optional[int]:
    if value is None:
        return None
    if isinstance(value, float) and np.isnan(value):
        return None
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def interval_coverage_percent(intervals: Sequence[Tuple[int, int]], total_len: int) -> float:
    """Padengimo dalis % (indeksai laikomi įtrauktiniais: +1)."""
    if total_len <= 0:
        return 0.0
    covered = sum(max(0, end - start + 1) for start, end in intervals)
    return (covered / total_len) * 100.0


def fmt_pct(x: Optional[float | int]) -> str:
    return f"{x:.1f}%" if isinstance(x, (float, int)) else "-"


def read_filenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.npy` failų ir visą DF (meta duomenims paimti)."""
    logging.info("Skaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "filename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'filename' and 'tag'")

    filtered = df[df["tag"] != "9999"]
    names = []
    for s in filtered["filename"].dropna():
        name = str(s).strip()
        if not name.endswith(".npy"):
            name += ".npy"
        names.append(name)

    logging.info("Atrinkta įrašų: %d", len(names))
    return names, df


def read_basenames_from_excel(xlsx_path: Path) -> tuple[list[str], pd.DataFrame]:
    """Skaito Excel, grąžina sąrašą `.xxx` failų ir visą DF (meta duomenims paimti)."""
    logging.info("Skaitomas Excel: %s", xlsx_path)
    df = pd.read_excel(xlsx_path, dtype=str)  # saugu tolimesnėms konversijoms
    if "basename" not in df.columns or "tag" not in df.columns:
        raise ValueError("Excel must contain columns: 'basename' and 'tag'")

    filtered = df[df["tag"] != "9999"]
    names = []
    for s in filtered["basename"].dropna():
        name = str(s)
        # if not name.endswith(".npy"):
        #     name += ".npy"
        names.append(name)

    logging.info("Atrinkta įrašų: %d", len(names))
    return names, df

def normalize_basename(basename: str) -> str:
    if "." in basename:
        left, right = basename.split(".", 1)
        right = right.ljust(3, "0")[:3]
        return f"{left}.{right}"
    return basename + ".000"


# === Pagrindinis srautas ========================================================


def main() -> None:
    print("\n")
    setup_logging()

    # create 'results' if not existed
    # try:
    #     base_dir = Path(__file__).resolve().parent
    # except NameError:
    #     base_dir = Path.cwd()

    # results_dir = base_dir / "results"
    # results_dir.mkdir(parents=True, exist_ok=True)
    
    base_dir = REC_DIR
    results_dir = base_dir / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    
    results_dir = REC_DIR / "results"
    results_dir.mkdir(parents=True, exist_ok=True)
    
    print("\nĮRAŠŲ KOKYBĖS ĮVERTINIMAS NUSTATANT TRIUKŠMŲ FRAGMENTŲ KIEKĮ ĮRAŠUOSE")
    print("Surandamos išskirtys (outliers), rspragos (rdropouts) ir U-Net tipo triukšmai\n")
    logging.info("CONFIG=%s | THRESHOLD=%.3f", CONFIG, THRESHOLD)
    logging.info("Rezultatų aplankas: %s | Modelio aplankas: %s", results_dir, MODEL_DIR)
    logging.info("Model: %s | Noise log: %s.[txt|xlsx]", MODEL_FILE_NAME, NOISE_LOG_BASENAME)

    check_consistency(MODEL_FILE_NAME, NOISE_LOG_BASENAME, CONFIG)

    model = load_model_checked(MODEL_DIR / MODEL_FILE_NAME, SEGMENT_LENGTH)
    basenames, df_meta = read_basenames_from_excel(LIST_DIR / EXCEL_NAME)
    print(f"Meta duomenų įrašų skaičius Excel: {len(df_meta)} | Unikalių failų vardų: {df_meta['basename'].nunique()}")
    # print(df_meta.head(5))

    # Kaupiame rezultatus
    text_log: list[str] = []
    rows_xlsx: list[dict] = []

    total_tp_all = 0.0
    n_files = len(basenames)

    n_annotated_files = 0
    sum_atp_across_annotated = 0.0
    sum_tp_in_annotated_files = 0.0  # bendras tp (algoritmo) tik tuose įrašuose, kurie turi anotacijas

    for i, basename in enumerate(basenames, start=1):
        # print(type(basename), basename)
        basename = normalize_basename(basename)
        fpath = REC_DIR / basename
        # print(f"\n=== {i}/{n_files} | Apdorojamas įrašas: {basename} ===")
        # print(f"Failo kelias: {fpath}")
        
        json_basename = basename + ".json"
        json_path = REC_DIR / json_basename
        # print(f"json failo kelias: {json_path}")
        
        try:
            ecg = get_ecg_signal(str(fpath))
            ann = get_ecg_noise_indices_annotated_ext(json_path)
            # print(f"ECG signalas užkrautas, ilgis: {len(ecg)} | Anotacijų kiekis: {len(ann) if ann is not None else 0}")
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"Failed to load {basename}: {exc}") from exc

        ecg_f = ecg_filter(ecg, fp)

        # (A) Anotuotų triukšmų padengimas
        if ann is not None:
            n_annotated_files += 1
            merged_ann = merge_lists_of_tuples(ann)
            atp = interval_coverage_percent(merged_ann, len(ecg_f))
            sum_atp_across_annotated += atp
            line3 = f"Anotuotų triukšmų kiekis: {len(ann)} | Anotuotų triukšmų dalis: {atp:.2f}%"
        else:
            atp = None
            line3 = "Anotuotų triukšmų nėra"

        # (B) Outliers + rdropouts + U-Net
        ecg_clean, out_idx, rdr_idx = find_outliers_rdropouts(ecg_f)
        _, noi_idx = run_ecg_denoising_pipeline(ecg_clean, model, CONFIG, THRESHOLD)

        merged_all = merge_lists_of_tuples([*noi_idx, *rdr_idx, *out_idx])
        tp = interval_coverage_percent(merged_all, len(ecg_f))
        total_tp_all += tp
        if ann is not None:
            sum_tp_in_annotated_files += tp

        # Meta duomenys iš Excel pagal stem
        # b3 = normalize_basename(basename)  # e.g. "1626933.710"

        mask = df_meta["basename"].astype(str).map(normalize_basename).eq(basename)
        # print(mask)
        row = df_meta.loc[mask]

        if not row.empty:
            filename = row.iloc[0].get("filename")
            quality = safe_int(row.iloc[0].get("quality"))
            noni = row.iloc[0].get("noni")
            tag = safe_int(row.iloc[0].get("tag"))
            print(f"Meta duomenys iš Excel: quality={quality} | noni={noni} | tag={tag}")
            mark = row.iloc[0].get("mark")
            N = row.iloc[0].get("N")
            S = row.iloc[0].get("S")
            V = row.iloc[0].get("V")
            comment = row.iloc[0].get("notes")
            line2 = f"quality:{quality} noni:{noni} tag:{tag} mark:{mark} N:{N} S:{S} V:{V} comment:{comment}"
        else:
            quality = tag = None
            noni = mark = N = S = V = comment = None
            line2 = f"Filename {basename} not found."
            filename = None

        line1 = (
            f"\n{i}. {basename} | outliers:{len(out_idx)} rdropouts:{len(rdr_idx)} "
            f"kiti triukšmai:{len(noi_idx)} | triukšmų dalis: {tp:.2f}%"
        )
        print(line1)
        print(line2)
        if ann is not None:
            print(line3)

        text_log.extend([line1, line2])
        if ann is not None:
            text_log.append(line3)

        rows_xlsx.append(
            {
                "fn": filename,
                "bn": basename,
                "qlt": quality,
               "tag": str(tag) if tag is not None else None,
                "out": len(out_idx),    # išskirtys (outliers), intervalų skaičius
                "rdr": len(rdr_idx),    # rspragos (rdropouts), intervalų skaičius
                "noi": len(noi_idx),    # judesio (motions), intervalų skaičius
                "tp": round(tp, 1),     # triukšmų padengimo dalis %
                "anoi": len(ann) if ann is not None else None,  # anotacijų kiekis
                "atp": round(atp, 1) if atp is not None else None,  # anotuotų triukšmų padengimo dalis %
            }
        )

    # Išsaugojimas: TXT + XLSX
    results_dir.mkdir(parents=True, exist_ok=True)

    noise_txt = results_dir / f"{NOISE_LOG_BASENAME}.txt"
    with open(noise_txt, "w", encoding="utf-8") as f:
        for line in text_log:
            f.write(line + "\n")
    print(f"\nTxt noise_log written to: {noise_txt}")
    # print(rows_xlsx[:3])  # peržiūrai keli pirmi įrašai

    df_log = pd.DataFrame(rows_xlsx)
    noise_xlsx = results_dir / f"{NOISE_LOG_BASENAME}.xlsx"
    df_log.to_excel(noise_xlsx, index=False)
    print(f"Excel noise_log written to: {noise_xlsx}")

    # Summary
    summary_lines: list[str] = []
    summary_lines.append("\nVISŲ ĮRAŠŲ REZULTATAI:")
    summary_lines.append(f"Visų analizuotų įrašų skaičius: {n_files}")

    avg_tp_all = (total_tp_all / n_files) if n_files else 0.0
    summary_lines.append(f"Vidutinė triukšmo dalis (tp) įrašuose: {avg_tp_all:.1f}%")

    summary_lines.append("Vidutinė triukšmo dalis 'tp' per 'quality':")
    for q in (0, 1, 2):
        vals = df_log.loc[df_log["qlt"] == q, "tp"]
        avg_q = round(vals.mean(), 1) if not vals.empty else None
        summary_lines.append(f"quality {q}: {fmt_pct(avg_q)}")

    summary_lines.append("\nANOTUOTŲ ĮRAŠŲ REZULTATAI:")
    summary_lines.append(f"Anotuotų įrašų skaičius: {n_annotated_files}")

    avg_atp = (sum_atp_across_annotated / n_annotated_files) if n_annotated_files else 0.0
    avg_tp_in_annot = (
        (sum_tp_in_annotated_files / n_annotated_files) if n_annotated_files else 0.0
    )
    summary_lines.append(f"Vidutinė anotuotų triukšmų dalis 'atp': {avg_atp:.1f}%")
    summary_lines.append(
        "Vidutinė (bendrų) triukšmų dalis 'tp' anotuotuose įrašuose: "
        f"{avg_tp_in_annot:.1f}%"
    )

    summary_path = results_dir / f"summary{MARKER}.txt"
    with open(summary_path, "w", encoding="utf-8") as f:
        for line in summary_lines:
            f.write(line + "\n")

    print(f"Noise_summary written to: {summary_path}")
    print("\n=== FINAL SUMMARY ===")
    with open(summary_path, "r", encoding="utf-8") as f:
        print(f.read())


if __name__ == "__main__":
    main()



PROJECT ROOT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET
PROJECT DIR: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/0_SELECT_ZIVE_DATA_2023



ĮRAŠŲ KOKYBĖS ĮVERTINIMAS NUSTATANT TRIUKŠMŲ FRAGMENTŲ KIEKĮ ĮRAŠUOSE
Surandamos išskirtys (outliers), rspragos (rdropouts) ir U-Net tipo triukšmai

13:00:34 | INFO | CONFIG={'FS': 200, 'SEGMENT_LENGTH': 1024, 'OVERLAP': 0.5} | THRESHOLD=0.080
13:00:34 | INFO | Rezultatų aplankas: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/0_SELECT_ZIVE_DATA_2023/AtsisiuntimasZiveDuomenu/DuomenysTestui/results | Modelio aplankas: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/MODEL_UNET
13:00:34 | INFO | Model: resunet_ecg_1024_0_5_3_7.keras | Noise log: noise_log_1024_0_5_3_7.[txt|xlsx]
13:00:34 | INFO | Suderinamumas OK | sufiksas:_1024_0_5_3_7 | SEGMENT_LENGTH:1024 | OVERLAP:0_5
13:00:34 | INFO | Kraunamas modelis: /home/kesju/DI/2025_ZIVEO/PROJECT_TRAIN_UNET/MODEL_UNET/resunet_ecg_1024_0_5_3_7.keras
13:00:38 | INFO | Modelis OK, input_shape: (None, 102